# Generator Demo

This notebook walks through the core features of the `Generator` processor.

## 1. Setup

In [1]:
# ! pip install kaggle kagglehub

In [2]:
# Standard
from pathlib import Path

# Third-party
import kagglehub
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel

# Homogene
from homogene import Generator, GenerativeModel

_ = load_dotenv()

Load the [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews) dataset from Kaggle.

In [3]:
def load_dataset() -> pd.DataFrame:
    """Load the Women's E-Commerce Clothing Reviews dataset from Kaggle.

    Returns:
        `pd.DataFrame` of the dataset.
    """
    dataset_dir = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
    dataset_path = Path(dataset_dir) / "Womens Clothing E-Commerce Reviews.csv"

    df = pd.read_csv(dataset_path, index_col=0)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df = df.dropna(subset=["review_text"])

    return df

reviews_df = load_dataset()

reviews_df.head()

,clothing_id,age,title,review_text,rating,recommended_ind,positive_feedback_count,division_name,department_name,class_name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


## 2. Model

Configure the `model` to pass to `Generator`.

**Option 1:** Use a plain string for the simplest case.

In [4]:
model = "gpt-5.5"

**Option 2:** Use a `GenerativeModel` instance for control over parameters like `temperature`, `max_tokens`, etc.

In [5]:
model = GenerativeModel(
    "gpt-5.5",
    temperature=1.0,
    max_tokens=50,
)

## 3. Freeform Generation

In [6]:
reviews_without_title_df = reviews_df[reviews_df["title"].isna()].copy()

print(f"{len(reviews_without_title_df):,} ({len(reviews_without_title_df)/len(reviews_df):.0%}) reviews are missing a title.")

2,966 (13%) reviews are missing a title.


Generate a title for each review that is missing one. `Generator` applies a plain-text instruction to every row **independently** and in **parallel**.

In [7]:
cleaner = Generator(
    df=reviews_without_title_df,
    columns=["review_text", "age"],
    instruction="Generate a title for the review in three words or less.",
    model=model,
)

Use `get_prompt()` to inspect the exact messages sent to the model.

In [8]:
prompt = cleaner.get_prompt(0)

print(prompt)

[
  {
    "role": "system",
    "content": "Generate a title for the review in three words or less."
  },
  {
    "role": "user",
    "content": "{\"review_text\": \"Absolutely wonderful - silky and sexy and comfortable\", \"age\": \"33\"}"
  }
]


Use `run()` to process the `pd.DataFrame`. Optionally, set `n` to limit to a subset first and validate output before committing to the full dataset.

In [9]:
_ = cleaner.run(n=5)

100%|██████████| 5/5 [00:04<00:00,  1.19it/s]

5/5 rows succeeded (100%)


The `result` attribute gives access to the _generation_ and _error_ columns after running.

In [10]:
cleaner.result

,generation,error,duration,cost,input_tokens,output_tokens
0,Silky Comfort,None,3.974976,0.001110,42,30
1,Pretty Midi Dress,None,4.393893,0.001430,112,29
11,Pretty and Flattering,None,3.942419,0.001295,43,36
30,Cute Flared Crops,None,4.031736,0.001150,56,29
36,Versatile Work Skirt,None,2.946617,0.001195,65,29


In [17]:
cleaner.summary

AttributeError: 'Generator' object has no attribute 'summary'

Increase `n` gradually to validate on larger subsets before running on the full dataset. Optionally, use `save_path` and `save_step` to checkpoint progress to disk periodically.

In [ ]:
reviews_df["generated_title"] = cleaner.run(n=20, 
                                            save_path='generated_reviews.csv', 
                                            save_step=10)

A value has now been added to the *'generated_title'* column for every row where *'title'* is missing.

In [ ]:
reviews_df[['review_text', 'title', 'generated_title']].head(20)

## 4. Structured Generation

Define a Pydantic `BaseModel`.

In [ ]:
class Review(BaseModel):
    sentiment: str
    keywords: list[str]

Pass it as `output_schema` to classify the sentiment and extract keywords from every review.

In [ ]:
analyzer = Generator(
    df=reviews_df,
    columns=["review_text", "rating"],
    instruction=(
        "Analyze this customer review. Classify the overall sentiment as 'positive', 'neutral', or 'negative'. Extract up to 5 keywords that capture the main topics mentioned."
    ),
    model=model,
    output_schema=Review,
)

analyzer.get_prompt(0)

In [ ]:
_ = analyzer.run(n=20)

In [ ]:
analyzer.result.head()

Expand the fields in the `Review` column into separate `pd.DataFrame` columns.

In [ ]:
reviews_df["sentiment"] = analyzer.result['generation'].map(lambda x: x.sentiment if x is not None else None)
reviews_df["keywords"]  = analyzer.result['generation'].map(lambda x: x.keywords if x is not None else None)

reviews_df[["review_text", "rating", "sentiment", "keywords"]].head()

## 5. Error Handling

Demonstrate both error modes using an invalid API key.

In [ ]:
faulty_model = GenerativeModel("gpt-5.5", api_key="invalid-key")

faulty = Generator(
    df=reviews_df.head(),
    columns=["review_text"],
    instruction="Write a title for this review in three words or less.",
    model=faulty_model,
)

**Case 1:** `on_error = "none"` <span style="color: grey;">_(default)_</span>

Errors are silently captured. Failed rows store `None` in the _generation_ column and the error message in the _error_ column. The run never crashes.

In [ ]:
try:
    faulty.run(n=5, on_error="none")
    faulty.result.head()

except Exception as e:
    print(f"Error: {e}")

**Case 2:** `on_error = "raise"`

The first exception is raised immediately, stopping the run.

In [ ]:
try:
    faulty.run(n=5, on_error="raise")
    faulty.result.head()

except Exception as e:
    print(f"Error: {e}")


## 6. Other Use Cases

Other use cases for `Generator`:
- Extract structured information from unstructured text
- Run an LLM-as-a-judge on each row